Ноутбук служит основой для изучения, однако многое нужно будет поискать и почитать самостоятельно. Никто вас не ограничивает в наведении красоты, но требуется выполнение того, что в задании.

В этом задании вам как и раньше нужно выполнить задания, но есть части, где нужно дополнить код и написать **выводы**.
***

In [1]:
import numpy as np
import pandas as pd
import gensim
import matplotlib.pyplot as plt

from nltk.tokenize import WordPunctTokenizer
from nltk.tokenize import TweetTokenizer
from gensim.models import Word2Vec
import gensim.downloader as api
from gensim.models import KeyedVectors

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.metrics.pairwise import cosine_similarity

import bokeh.models as bm, bokeh.plotting as pl
from bokeh.io import output_notebook

# «Теоретическая часть» (3 pt)

В прошлом задании вы поработали с токенизацией и уже неплохо понимаете, что это такое и как ее можно использовать. В этой же домашке мы пойдем дальше и поработаем с **векторным представлением естественного языка**.

Векторному представлению можно посвятить много времени, но наша задача — разобраться в том, как что-то можно использовать, а не погружаться в это слишком глубоко.

***

Вообще подходы к векторному представлению токенов (_тут уже поменяю на токены, вы знаете, что это необязательно слова_) можно очень грубо разделить на две группы (так то их очень [много](https://habr.com/ru/articles/515036/)):
1. **count-base** подходы (насчитывать какие-то статистики и использовать их),
2. **«обучаемые»** векторы (обычно нейросетевые подходы).

> Во второй случай входит очень много чего: это и, отчасти, скрытые представления трансформеров, и какие-то сложные внутренние части моделей. Нас будут интересовать именно те подходы, которые позволяют работать с такими векторами токенов (**эмбеддингами**) как с отдельными сущностями и получать из них информацию.

Среди первых самыми известными и «зафиксированными» являются подходы [**Bow (Bag of words)**](https://ru.wikipedia.org/wiki/%D0%9C%D0%B5%D1%88%D0%BE%D0%BA_%D1%81%D0%BB%D0%BE%D0%B2) и [**TF-IDF (Term Frequency — Inverse Document Frequency)**](https://habr.com/ru/companies/otus/articles/755772/). В теоретической части мы их касаться не будем, так как они весьма очевидны, но вот для практической повторите.  
Из вторых мы поговорим только про три популярных и довольно простых метода:
[**Word2Vec**](https://habr.com/ru/articles/446530/),
[**GloVe**](https://en.wikipedia.org/wiki/GloVe) и
[**FastText**](https://www.geeksforgeeks.org/machine-learning/fasttext-working-and-implementation/).

_То, как они работают оставляю на самостоятельное изучение, в этой домашке это не является фокусом, главное — их использование._

И так, начнем с наблюдения сути и работы этих методов.

## Обработка текста (1 pt)

Начнем как обычно с **загрузки данных**, вы уже их знаете, это тот же самый датасет комментариев к видео.

In [2]:
df = pd.read_csv('100000 Youtube comments sentiment dataset.csv').dropna().reset_index(drop=True)
df = df[df['Sentiment'] != 'neutral']

In [3]:
# соединим все в текст, тут нам тональность не нужна, нужен только текст
data = list(df['Comment'])
print(data[:3])

['New fear of elevators unlocked 😖', 'My heart stopped when he almost dropped the iPad 😂', 'That glass brige prank is dangerous. Could make someone fall off the bridge']


In [4]:
# TODO: поэксперементируйте с индексом и найдите интересное предложение с пунктуацией
data[42]

'Glass bridge ruled. Absolutely brilliant!\n❤️👏👏'

### Токенизация

В этом задании мы используем **токенизацию по словам**, чтобы можно было интерпретировать смысл (_в прошлый раз вы скорее всего поняли, что если не слова, то сильно смысла не заметишь_). 

In [ ]:
tokenizer = WordPunctTokenizer()
# вставьте тот же индекс
print(tokenizer.tokenize(data[42]))

In [ ]:
# TODO: токенизируйте все и приведите в нижний регистр
#       должен получиться лист листов токенов каждого предложения
data_tok = [... for line in data]

In [ ]:
assert all(isinstance(row, (list, tuple)) for row in data_tok), "please convert each line into a list of tokens (strings)"
assert all(all(isinstance(tok, str) for tok in row) for row in data_tok), "please convert each line into a list of tokens (strings)"
is_latin = lambda tok: all('a' <= x.lower() <= 'z' for x in tok)
assert all(map(lambda l: not is_latin(l) or l.islower(), map(' '.join, data_tok))), "please make sure to lowercase the data"

In [ ]:
print(' '.join(data_tok[42]))

### Создание эмбеддингов

Теперь у нас есть отдельные **токены** и можем их переводить в числовое пространство. Для этого, как уже знаете, есть разные методы: `Word2Vec` и `GloVe` с разными целевыми функциями. Кроме того, есть `FastText`, который использует модели символьного уровня для обучения эмбеддингов.

Выбор огромен, поэтому давайте начнем с малого: `gensim` — это библиотека NLP, которая включает в себя множество векторных моделей, включая `word2vec`.

Самое прекрасное в таких векторах — это то, что они сохраняют семантическое и контекстное значение слова (_Вы точно пример про короля и королеву где-то видели_)! Не будем об этом долго говорить, давайте лучше посмотрим и поймем, что это для нас значит. 

In [ ]:
# тут мы обучаем модель, то есть сами создаем эмбеддинги
model = Word2Vec(data_tok, 
                 vector_size=32,      # размер вектора
                 min_count=5,         # минимальная встречаемость слова
                 window=5).wv         # контекстное окно

In [ ]:
# теперь можно получать векторы!
model.get_vector('anything')

Теперь мы можем поэксперементировать с преимуществами векторного представления! Мы можем посмотреть **похожие в векторном пространстве слова**, а также **поскладывать и по вычитать**!

In [ ]:
# TODO: попробуйте разные слова и найдите наиболее интересные!
model.most_similar("man")

In [ ]:
# TODO: попробуйте разные преобразования с разными парами
#       (это ищет похожее к видео без лайков)
model.most_similar(positive=["video"], negative=["likes"])

**ПАРУ СЛОВ О ТОМ, ЧТО ПОЛУЧИЛОСЬ**

_Согласитесь, что иногда получается какая-то ерунда... Все потому, что мы учились на весьма конкретном и небольшом корпусе, поэтому и слова для нас не просто слова, а слова именно в этом контексте._

### Использование предобученной модели

Чтобы векторы запоминали больше контекстного смысла, нужно много контекста) То есть нужны большие корпуса текстов и время на обучение. К счастью, в настоящее время вы можете получить предварительно обученную модель эмбеддингов в 2 строки кода.

> После первой загрузки (или при удалении вручную) модель сохраняется в каталоге ~/gensim_data или %USER_PATH%/gensim_data. Это можно проверить, установив для параметра return_path значение True.

In [ ]:
# используем другую модель
model_loaded = api.load('glove-twitter-100')

In [ ]:
# TODO: проделайте те же эксперименты, что и в прошлой части (слова и пары)
model_loaded.most_similar(positive=["coder", "money"], negative=["brain"])

**ПАРУ СЛОВ О ТОМ, ЧТО ПОЛУЧИЛОСЬ**

_Теперь мы с одной стороны умеем переводить текст в числа, а с другой — операции с числами что-то интерпретируемое значат как операции со словами._

## Визуализация данных при помощи эмбеддингов (2 pt)

Один из способов узнать, хороши ли наши векторы, — это построить их график. Дело в том, что эти векторы находятся в пространстве более `30D`, а мы, люди, больше привыкли к `2-3D`.

К счастью, мы, изучающие компьютерные технологии, знаем о методах **уменьшения размерности**. Давайте используем это для составления списка из 1000 наиболее часто встречающихся слов.

In [ ]:
words = model.index_to_key[:1000] # это модель, которую мы обучили
                                  # параллельно проделайте все то же для загруженной модели!

print(words[::100])

In [ ]:
# TODO: для каждого слова посчитайте вектор
word_vectors = np.array(...) 

In [ ]:
assert isinstance(word_vectors, np.ndarray)
assert word_vectors.shape == (len(words), 32)
assert np.isfinite(word_vectors).all()

### PCA

По классике сначала визуализируем при помощи линейного метода

In [ ]:
pca = PCA(n_components=2)
# TODO: обучите pca на word_vectors
word_vectors_pca = ...
# TODO: отнормируйте векторы
word_vectors_pca = ...

In [ ]:
assert word_vectors_pca.shape == (len(word_vectors), 2), "there must be a 2d vector for each word"
assert max(abs(word_vectors_pca.mean(0))) < 1e-5, "points must be zero-centered"
assert max(abs(1.0 - word_vectors_pca.std(0))) < 1e-2, "points must have unit variance"

In [ ]:
# нарисуем их
output_notebook()

def draw_vectors(x, y, radius=10, alpha=0.25, color='blue',
                 width=600, height=400, show=True, **kwargs):
    if isinstance(color, str): color = [color] * len(x)
    data_source = bm.ColumnDataSource({ 'x' : x, 'y' : y, 'color': color, **kwargs })

    fig = pl.figure(active_scroll='wheel_zoom', width=width, height=height)
    fig.scatter('x', 'y', size=radius, color='color', alpha=alpha, source=data_source)

    fig.add_tools(bm.HoverTool(tooltips=[(key, "@" + key) for key in kwargs.keys()]))
    if show: pl.show(fig)
    return fig

In [ ]:
draw_vectors(word_vectors_pca[:, 0], word_vectors_pca[:, 1], token=words)

**ВИЗУАЛИЗИРУЙТЕ ТАКЖЕ И ДЛЯ ЗАГРУЖЕННОЙ МОДЕЛИ!**

Видны ли группы слов? Можно ли как-то их объяснить?  
**ОТВЕТ**

### t-SNE

Как могли заметить в прошлых домашках, **нелинейные методы круче кластеризуют**, поэтому давайте их попробуем.

In [ ]:
# TODO: обучите tsne на word_vectors
word_tsne = ...

In [ ]:
draw_vectors(word_tsne[:, 0], word_tsne[:, 1], color='green', token=words)

**ВИЗУАЛИЗИРУЙТЕ ТАКЖЕ И ДЛЯ ЗАГРУЖЕННОЙ МОДЕЛИ!**

Стали ли группы лучше разделяться? Можно ли как-то объяснить новые группы (если они появились)?  
**ОТВЕТ**

_На этом мы закончим с изучением векторов и перейдем к пррррактике!_

# Практическая часть (7 pt)

В этой части вы (мы) повторите задачу прошлой домашки (классификация комментариев). Мы решим эту проблему, используя как **классические методы**, так и подход, основанный на **эмбеддингах**.

_Тут можно рестартнуть ядро, чтобы случаем что-то не затереть и не переиспользовать. Очевидно, что порядок ячеек собъется, в этой домашке норм, если следующая ячейка будет второй и дальше все будет по порядку._

In [2]:
df = pd.read_csv('100000 Youtube comments sentiment dataset.csv').dropna().reset_index(drop=True)
df = df[df['Sentiment'] != 'neutral']
data = df.sample(1_000, random_state=42) # мы уже знаем, что и нашими сууупер простыми методами метрики норм, поэтому возьмем мало данных)

texts = data['Comment'].values
target = data['Sentiment'].values
target = (target == 'Negative').astype(int)

print(pd.Series(target).value_counts())

1    507
0    493
Name: count, dtype: int64


In [ ]:
# разделим на трейн и тест, сид и размер не менять,
texts_train, texts_test, y_train, y_test = train_test_split(texts, target, test_size=0.4, random_state=42)

### Обработка и токенизация (0,5 pt)

In [ ]:
tokenizer = TweetTokenizer()
preprocess = lambda text: ' '.join(tokenizer.tokenize(text.lower()))

# TODO: обработайте каждое предложение в трейне и тесте

texts_train = ...
texts_test = ...

In [ ]:
assert texts_train[42] ==  "i'm pretty sure a lot of people misses the bro fist wamen and men alike"
assert texts_test[42] == 'a 20lbs girl challenges a badass kickboxer 😂'
assert len(texts_test) == len(y_test)

## BoW (2 pt)

Теперь, когда мы подготовили данные, мы можем переходить к решению задачи.

Один из традиционных подходов к решению заключается в использовании фичей «мешка слов»:
* составьте **словарь часто встречающихся слов** (используйте только обучающие данные),
* для каждого примера из обучающей выборки, подсчитайте, **сколько раз в нем встречается слово** (для каждого слова в словаре),
* считайте этот подсчет **признаками для некоторого классификатора**.

Это можно сделать при помощи реализованных функций, но тут **реализуйте это самостоятельно**.

_Это очень похоже на то, что вы уже делали, только теперь не 0/1, а нормальные числа)_

In [ ]:
# TODO: найдите k наиболее часто встречающихся слов в texts_train,
#       отсортируйте их по количество встречаемости (по убыванию)
k = 1000

all_words = [word for text in texts_train for word in text.split()]
bow_vocabulary_dict = {}
...
        
sorted_vocabulary = sorted(bow_vocabulary_dict.items(), key=lambda x: x[1], reverse=True)
bow_vocabulary = [word for word, count in sorted_vocabulary[:k]]

print('example features:', sorted(bow_vocabulary)[::100])

In [ ]:
#TODO: реализуйте функцию, которая преобразует текст в вектор
#      вектор размера словаря, на месте индекса слова стоит количество раз, которое оно встречается в тексте

def text_to_bow(text):
    vector = np.zeros(len(bow_vocabulary), dtype=np.float32)
    ...
    
    return vector

In [ ]:
X_train_bow = np.stack(list(map(text_to_bow, texts_train)))
X_test_bow = np.stack(list(map(text_to_bow, texts_test)))

In [ ]:
k_max = len(set(' '.join(texts_train).split()))
assert X_train_bow.shape == (len(texts_train), min(k, k_max))
assert X_test_bow.shape == (len(texts_test), min(k, k_max))
assert len(bow_vocabulary) <= min(k, k_max)
assert X_train_bow[6, bow_vocabulary.index('.')] == texts_train[6].split().count('.')

Также как и в прошлый раз используем простую модель

In [ ]:
# TODO: подберите подходящие параметры модели
bow_model = LogisticRegression().fit(X_train_bow, y_train)

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

for name, X, y, model in [
    ('train', X_train_bow, y_train, bow_model),
    ('test ', X_test_bow, y_test, bow_model)
]:
    proba = model.predict_proba(X)[:, 1]
    auc = roc_auc_score(y, proba)
    plt.plot(*roc_curve(y, proba)[:2], label='%s AUC=%.4f' % (name, auc))

plt.plot([0, 1], [0, 1], '--', color='black',)
plt.legend(fontsize='large')
plt.grid()

test_accuracy = np.mean(bow_model.predict(X_test_bow) == y_test)
print(f"Model accuracy: {test_accuracy:.3f}")
assert test_accuracy >= 0.9, "Hint: tune the parameter C to improve performance"
print("Well done!")

_Не так уж и плохо, правда?) А лучше сможем?_

## TF-IDF (2 pt)

Попробуем применить другой метон создания фич. Интуиция у него такая:

Не все слова одинаково полезны. Можно расставить приоритеты по редким словам и словам пониженного уровня, таким как "и"/"или", используя функции **tf-idf**. Эта аббревиатура расшифровывается как **частота текста/обратная частота документа** и означает именно это:

$$ \text{feature}_i = \frac{\text{Count}(word_i \in x)}{\text{Total number of words in } x} \times \log\left(\frac{N}{\text{Count}(word_i \in D) + \alpha}\right),$$

где $x$ — отдельный текст, $D$ — ваш набор данных (коллекция текстов), $N$ — общее количество документов, а $\alpha$ — гиперпараметр сглаживания (обычно 1). А $Count(word_i \in D)$ - это количество документов, в которых фигурирует $word_i$.

Также может быть хорошей идеей нормализовать каждый пример после вычисления функций tf-idf.

In [ ]:
# TODO: реализовать TF-IDF, размерность оставить как в прошлом примере

vocabulary_count_dict = {word: count for word, count in sorted_vocabulary[:k]}
N = len(texts_train)
alpha = 1

def text_to_tf_idf(text):
    vector = np.zeros(len(vocabulary_count_dict), dtype=np.float32)
    text_vocabulary_count_dict = {}
    ...
        return vector

In [ ]:
X_train_tfidf = np.stack(list(map(text_to_tf_idf, texts_train)))
X_test_tfidf = np.stack(list(map(text_to_tf_idf, texts_test)))

In [ ]:
tfidf_model = LogisticRegression(random_state=42).fit(X_train_tfidf, y_train) 

In [ ]:
for name, X, y, model in [
    ('train', X_train_tfidf, y_train, tfidf_model),
    ('test ', X_test_tfidf, y_test, tfidf_model)
]:
    proba = model.predict_proba(X)[:, 1]
    auc = roc_auc_score(y, proba)
    plt.plot(*roc_curve(y, proba)[:2], label='%s AUC=%.4f' % (name, auc))

plt.plot([0, 1], [0, 1], '--', color='black',)
plt.legend(fontsize='large')
plt.grid()

test_accuracy = np.mean(tfidf_model.predict(X_test_tfidf) == y_test)
print(f"Model accuracy: {test_accuracy:.3f}")

**ЗАДАНИЕ**: сравните полученные модели и метрики, какие минусы у этих моделей?

## Fasttext (1,5 pt)

Пойдем дальше и используем векторизацию, которая оставляет семантический смысл предложений!

Вместо подсчета частот для каждого слова, мы сопоставим все слова с **заранее подготовленными векторами** слов и **усредним по ним**, чтобы получить текстовые характеристики предложения. 

* Во-первых, мы так сможем работать даже с теми словами, которых нет в обучающей выборке.
* Во-вторых, мы можем работать в меньшей размерности.

Как и ранее будем использовать предобученную модель для получения эмбеддингов.

In [ ]:
embeddings = api.load('fasttext-wiki-news-subwords-300')

In [ ]:
# TODO: реализуйте функцию, которая преобразует текст в сумму векторов токенов

def vectorize_sum(text):
    embedding_dim = embeddings.vectors.shape[1]
    features = np.zeros([embedding_dim], dtype='float32')
    
    ...
    
    return features

assert np.allclose(
    vectorize_sum("who cares anymore . they attack with impunity .")[::70],
    np.array([ 0.0108616 ,  0.0261663 ,  0.13855131, -0.18510573, -0.46380025])
)

In [ ]:
X_train_ft = np.stack([vectorize_sum(text) for text in texts_train])
X_test_ft = np.stack([vectorize_sum(text) for text in texts_test])

In [ ]:
ft_model = LogisticRegression(random_state=42).fit(X_train_ft, y_train) # поэксперементируйте с параметрами

for name, X, y, model in [
    ('bow train', X_train_bow, y_train, bow_model),
    ('bow test ', X_test_bow, y_test, bow_model),
    ('vec train', X_train_ft, y_train, ft_model),
    ('vec test ', X_test_ft, y_test, ft_model)
]:
    proba = model.predict_proba(X)[:, 1]
    auc = roc_auc_score(y, proba)
    plt.plot(*roc_curve(y, proba)[:2], label='%s AUC=%.4f' % (name, auc))

plt.plot([0, 1], [0, 1], '--', color='black',)
plt.legend(fontsize='large')
plt.grid()

test_accuracy = np.mean(ft_model.predict(X_test_ft) == y_test)
print(f"Model accuracy: {test_accuracy:.3f}")

_Ну как?_

## Использование не для машинного обучения (1 pt)

Пока мы с вами поиспользовали векторы предложений только для обучения, но давайте попробуем использовать и для «визуального» анализа наших данных!

Давайте попробуем научиться определять **наиболее похожее по смыслу предложение** из нашего корпуса (или топ таких)!

Для этого будем задавать смысл в виде строки, считать его вектор, а затем по **косиносному расстоянию** искать самые похожие предложения в данных. Это практически то же, что мы делали и в теоретичекой части.

In [ ]:
qwery = "It's the most beautiful thing I've seen."
qwery = ' '.join(tokenizer.tokenize(qwery.lower()))

bow_vec = text_to_bow(qwery)
tfidf_vec = text_to_tf_idf(qwery)
ft_vec = vectorize_sum(qwery)

# посмотрите на значения векторов, что думаете?

In [ ]:
def find_similar_comments(query_vec, comment_embeddings, comment_texts=texts_train, top_n=5):
    
    query_vec = query_vec.reshape(1, -1)  # для совместимости с cosine_similarity
    
    similarities = ...
    top_indices = ...
    
    results = []
    for idx in top_indices:
        results.append(...)
    
    return results

In [ ]:
bow_sims = find_similar_comments(bow_vec, X_train_bow)

for res in bow_sims:
    print(f"{res['text']} | similarity = {res['similarity']:.4f}\n")

In [ ]:
tfidf_sims = find_similar_comments(tfidf_vec, X_train_tfidf)

for res in tfidf_sims:
    print(f"{res['text']} | similarity = {res['similarity']:.4f}\n")

In [ ]:
ft_sims = find_similar_comments(ft_vec, X_train_ft)

for res in ft_sims:
    print(f"{res['text']} | similarity = {res['similarity']:.4f}\n")

**ВОПРОС**: что лучше подходит, можно ли как-то это использовать при анализе текстовых данных?

***
_ну все, конец_